# Лабораторная работа: Apache Spark

Запустите ячейки сверху вниз. Полученные метрики внесите в вывод.

In [ ]:
%pip -q install pyspark

In [ ]:
from google.colab import files
from pyspark.sql import SparkSession,functions as F
from pyspark.ml.feature import VectorAssembler,Imputer
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression,DecisionTreeClassifier,RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import pandas as pd
u=files.upload();name=next(n for n in u if n.endswith('.csv'))
spark=SparkSession.builder.master('local[*]').getOrCreate()
df=spark.read.csv(name,header=True,inferSchema=True)
df.printSchema();df.show(5)
df.groupBy('Fire_Alarm').count().show()
features=[c for c,t in df.dtypes if t in ('int','bigint','double','float') and c!='Fire_Alarm']
data=df.select(*[F.col(c).cast('double').alias(c) for c in features],F.col('Fire_Alarm').cast('double').alias('label')).dropna(subset=['label'])
train,test=data.randomSplit([.8,.2],seed=42)
results=[]
for name,model in [('LogisticRegression',LogisticRegression(maxIter=40)),('DecisionTree',DecisionTreeClassifier(seed=42)),('RandomForest',RandomForestClassifier(numTrees=30,seed=42))]:
    pipeline=Pipeline(stages=[Imputer(inputCols=features,outputCols=[c+'_imp' for c in features]),VectorAssembler(inputCols=[c+'_imp' for c in features],outputCol='features'),model])
    pred=pipeline.fit(train).transform(test)
    results.append({'model':name,**{m:MulticlassClassificationEvaluator(metricName=m).evaluate(pred) for m in ('accuracy','f1','weightedPrecision','weightedRecall')}})
display(pd.DataFrame(results))